# Movie Recommendation System
Content-based recommender comparing **TF-IDF** vs **BGE Sentence Embeddings**, built on `movies_metadata.csv`, `credits.csv`, and `keywords.csv` (Kaggle: The Movies Dataset).

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
import warnings
warnings.filterwarnings('ignore')

### import/pre-process & clean dataset

In [4]:
df = pd.read_csv('movies_metadata.csv')
credits = pd.read_csv('credits.csv')
keywords = pd.read_csv('keywords.csv')

print(df.shape, credits.shape, keywords.shape)

(45466, 24) (45476, 3) (46419, 2)


In [6]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [7]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [9]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [10]:
# Drop fully-duplicate rows first
df = df.drop_duplicates().reset_index(drop=True)

# id must be numeric and clean before merging
df['id'] = pd.to_numeric(df['id'], errors='coerce')
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
keywords['id'] = pd.to_numeric(keywords['id'], errors='coerce')

df = df.dropna(subset=['id'])
credits = credits.dropna(subset=['id'])
keywords = keywords.dropna(subset=['id'])

df['id'] = df['id'].astype(int)
credits['id'] = credits['id'].astype(int)
keywords['id'] = keywords['id'].astype(int)

In [11]:
df = df.drop_duplicates(subset='id', keep='first')
credits = credits.drop_duplicates(subset='id', keep='first')
keywords = keywords.drop_duplicates(subset='id', keep='first')

print(df.shape, credits.shape, keywords.shape)

(45433, 24) (45432, 3) (45432, 2)


In [18]:
df = df.merge(credits, on='id', how='left').merge(keywords, on='id', how='left')

print(df.shape)
df.columns

(45430, 10)


Index(['id', 'title', 'overview', 'genres', 'tagline', 'vote_average',
       'popularity', 'cast', 'crew', 'keywords'],
      dtype='object')

In [19]:
df = df[['id', 'title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity','cast', 'crew', 'keywords']]

# Can't recommend a movie with no title
df = df.dropna(subset=['title'])

df['overview'] = df['overview'].fillna(' ')
df['tagline'] = df['tagline'].fillna(' ')

In [20]:
df.isnull().sum()

id              0
title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
cast            1
crew            1
keywords        1
dtype: int64

In [22]:
import ast

def safe_parse_names(x):
    """Extract 'name' fields from a JSON-like string. Returns '' for NaN or malformed input."""
    if pd.isna(x):
        return ''
    try:
        return " ".join(i['name'] for i in ast.literal_eval(x))
    except (ValueError, SyntaxError):
        return ''

df['genres'] = df['genres'].apply(safe_parse_names)
df['keywords'] = df['keywords'].apply(safe_parse_names)

print(df['genres'][0])
print(df['keywords'][0])
assert (df['genres'] == '').sum() < len(df) * 0.5, "genres mostly empty — check source column before proceeding"
assert (df['keywords'] == '').sum() < len(df) * 0.5, "keywords mostly empty — check source column before proceeding"

Animation Comedy Family
jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life


In [23]:
def get_director(crew_str):
    if pd.isna(crew_str):
        return ''
    try:
        crew = ast.literal_eval(crew_str)
        for person in crew:
            if person.get('job') == 'Director':
                return person['name'].replace(" ", "")
        return ''
    except (ValueError, SyntaxError):
        return ''

def get_top_cast(cast_str, top_n=3):
    if pd.isna(cast_str):
        return ''
    try:
        cast = ast.literal_eval(cast_str)
        return " ".join(c['name'].replace(" ", "") for c in cast[:top_n])
    except (ValueError, SyntaxError):
        return ''

df['director'] = df['crew'].apply(get_director)
df['cast'] = df['cast'].apply(get_top_cast)

print(df['cast'][0])
print(df['director'][0])

TomHanks TimAllen DonRickles
JohnLasseter


In [24]:
df['tags'] = (
    df['overview'] + " " +
    df['genres'] + " " +
    df['tagline'] + " " +
    df['keywords'] + " " +
    df['cast'] + " " +
    df['director']
)

In [25]:
print(df['tags'][0])

Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family   jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life TomHanks TimAllen DonRickles JohnLasseter


In [28]:
df['cast'][0]

'TomHanks TimAllen DonRickles'

In [26]:
df.head()

,id,title,overview,genres,tagline,vote_average,popularity,cast,crew,keywords,director,tags
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943,TomHanks TimAllen DonRickles,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",jealousy toy boy friendship friends rivalry bo...,JohnLasseter,"Led by Woody, Andy's toys live happily in his ..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539,RobinWilliams JonathanHyde KirstenDunst,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",board game disappearance based on children's b...,JoeJohnston,When siblings Judy and Peter discover an encha...
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129,WalterMatthau JackLemmon Ann-Margret,"[{'credit_id': '52fe466a9251416c75077a89', 'de...",fishing best friend duringcreditsstinger old men,HowardDeutch,A family wedding reignites the ancient feud be...
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495,WhitneyHouston AngelaBassett LorettaDevine,"[{'credit_id': '52fe44779251416c91011acb', 'de...",based on novel interracial relationship single...,ForestWhitaker,"Cheated on, mistreated and stepped on, the wom..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519,SteveMartin DianeKeaton MartinShort,"[{'credit_id': '52fe44959251416c75039ed7', 'de...",baby midlife crisis confidence aging daughter ...,CharlesShyer,Just when George Banks has recovered from his ...


In [27]:
df.isnull().sum()

id              0
title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
cast            0
crew            1
keywords        0
director        0
tags            0
dtype: int64

### NLP - text cleaning

In [30]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [31]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jangi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jangi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [32]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [33]:
def preprocess_text(text):
    text = str(text).lower() # lowercase
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation
    words = text.split() # tokenization
    words = [word for word in words if word not in stop_words] # remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words] # lemmatization  => running, ran, runs → run ; better → good

    return " ".join(words)

In [34]:
df['tags'] = df['tags'].apply(preprocess_text)

In [35]:
df = df.reset_index(drop=True)

print(df['tags'][0])

led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference animation comedy family jealousy toy boy friendship friend rivalry boy next door new toy toy come life tomhanks timallen donrickles johnlasseter


### Text Vectorization

In [36]:
indices = pd.Series(df.index, index=df['title'])
indices = indices[~indices.index.duplicated(keep='first')]

In [37]:
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Caged Heat 3000                45423
Subdue                         45425
Century of Birthing            45426
Satan Triumphant               45428
Queerama                       45429
Length: 42277, dtype: int64

In [38]:
print(len(indices), df.shape[0])

42277 45430


In [39]:
 # Tf-Idf
from sklearn.feature_extraction.text import TfidfVectorizer

In [40]:
tfidf = TfidfVectorizer(max_features = 50000, ngram_range = (1,2), stop_words = 'english')

In [41]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [42]:
print(tfidf_matrix.shape)
assert tfidf_matrix.shape[0] == df.shape[0]

(45430, 50000)


### Cosine Similarity

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

In [44]:
def recommend(title, n=10):
    if title not in indices:
        return ['Movie not found!']
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx = sim_score.argsort()[::-1][1:n + 1]  # argsort --> to sort according to index
    return df['title'].iloc[similar_idx]

recommend('The Dark Knight', 5)

18244         The Dark Knight Rises
10119                 Batman Begins
15505    Batman: Under the Red Hood
1328                 Batman Returns
150                  Batman Forever
Name: title, dtype: object

### Saving model

In [46]:
import pickle

pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))
pickle.dump(indices,open('indices.pkl','wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf,open('tfidf.pkl','wb'))

## Now using Embeddings (HuggingFaceEmbeddings)

In [47]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [48]:
embeddings = embed_model.encode(
    df['tags'].tolist(),
    show_progress_bar = True,
    batch_size = 64,
    normalize_embeddings = True  # required for BGE — cosine similarity assumes normalized vectors
)

Batches:   0%|          | 0/710 [00:00<?, ?it/s]

In [49]:
print(embeddings.shape)
assert embeddings.shape[0] == df.shape[0]

(45430, 384)


In [50]:
np.save('movie_embeddings.npy', embeddings)

In [51]:
def recommend_embedding(title, n=10):
    if title not in indices:
        return ['Movie not found!']
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    sim_score = cosine_similarity([embeddings[idx]], embeddings).flatten()
    similar_idx = sim_score.argsort()[::-1][1:n + 1]
    return df['title'].iloc[similar_idx]

In [52]:
recommend_embedding('The Dark Knight', 5)

18244                 The Dark Knight Rises
18027                      Batman: Year One
10119                         Batman Begins
31047    Batman v Superman: Dawn of Justice
1490                         Batman & Robin
Name: title, dtype: object

### Comarision

In [53]:
test_titles = ['The Dark Knight', 'Inception', 'Toy Story', 'The Godfather', 'Interstellar', 'Titanic']

for t in test_titles:
    print(f"\n=== {t} ===")
    print("TF-IDF:    ", recommend(t, 5).tolist())
    print("Embedding: ", recommend_embedding(t, 5).tolist())


=== The Dark Knight ===
TF-IDF:     ['The Dark Knight Rises', 'Batman Begins', 'Batman: Under the Red Hood', 'Batman Returns', 'Batman Forever']
Embedding:  ['The Dark Knight Rises', 'Batman: Year One', 'Batman Begins', 'Batman v Superman: Dawn of Justice', 'Batman & Robin']

=== Inception ===
TF-IDF:     ['Gamer', 'UFO - Distruggete base Luna!', 'Cypher', 'Minority Report', 'III']
Embedding:  ['Conspiracy Theory', 'Limitless', 'Cypher', 'The Gift', 'Dream Man']

=== Toy Story ===
TF-IDF:     ['Toy Story 2', 'Toy Story 3', 'Toy Story of Terror!', 'Small Soldiers', 'Silent Night, Deadly Night 5: The Toy Maker']
Embedding:  ['Toy Story 3', 'Toy Story 2', "Child's Play 3", 'Silent Night, Deadly Night 5: The Toy Maker', 'Golden Christmas 2']

=== The Godfather ===
TF-IDF:     ['The Godfather Trilogy: 1972-1990', 'The Godfather: Part II', 'Honor Thy Father', 'Election', 'Paradies 505. Ein Niederbayernkrimi']
Embedding:  ['The Godfather: Part II', 'The Godfather: Part III', 'Lucky Luciano',

In [54]:
def metadata_overlap_score(title, recommend_fn, field, n=10):
    if title not in indices:
        return None
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    input_vals = set(str(df.loc[idx, field]).split())
    recs = recommend_fn(title, n)
    scores = []
    for r in recs:
        if r not in indices:
            continue
        r_idx = indices[r]
        if isinstance(r_idx, pd.Series):
            r_idx = r_idx.iloc[0]
        rec_vals = set(str(df.loc[r_idx, field]).split())
        union = input_vals | rec_vals
        if not union:
            continue
        scores.append(len(input_vals & rec_vals) / len(union))
    return np.mean(scores) if scores else 0

In [56]:
import random
random.seed(42)
sample_titles = random.sample(list(indices.index), 50)

for field in ['genres', 'keywords', 'cast', 'director']:
    tfidf_scores = [metadata_overlap_score(t, recommend, field, 10) for t in sample_titles]
    embed_scores = [metadata_overlap_score(t, recommend_embedding, field, 10) for t in sample_titles]
    tfidf_avg = np.mean([s for s in tfidf_scores if s is not None])
    embed_avg = np.mean([s for s in embed_scores if s is not None])
    print(f"{field:10} — TF-IDF: {tfidf_avg:.3f}  |  Embedding: {embed_avg:.3f}")

genres     — TF-IDF: 0.361  |  Embedding: 0.390
keywords   — TF-IDF: 0.097  |  Embedding: 0.051
cast       — TF-IDF: 0.026  |  Embedding: 0.016
director   — TF-IDF: 0.026  |  Embedding: 0.022


In [57]:
def diversity_score(recs, matrix, indices):
    idxs = []
    for r in recs:
        if r not in indices:
            continue
        i = indices[r]
        if isinstance(i, pd.Series):
            i = i.iloc[0]
        idxs.append(i)
    if len(idxs) < 2:
        return 0
    vectors = matrix[idxs]
    if hasattr(vectors, 'toarray'):
        vectors = vectors.toarray()
    sim_matrix = cosine_similarity(vectors)
    n = len(sim_matrix)
    return (sim_matrix.sum() - n) / (n * (n - 1))

sample_check = random.sample(list(indices.index), 20)
tfidf_div = np.mean([diversity_score(recommend(t, 10).tolist(), tfidf_matrix, indices) for t in sample_check])
embed_div = np.mean([diversity_score(recommend_embedding(t, 10).tolist(), embeddings, indices) for t in sample_check])

print(f"TF-IDF avg pairwise similarity within recs:    {tfidf_div:.3f}")
print(f"Embedding avg pairwise similarity within recs: {embed_div:.3f}")

TF-IDF avg pairwise similarity within recs:    0.143
Embedding avg pairwise similarity within recs: 0.684
